In [ ]:
# Dicas para executar notebooks no Google Colab:
# `pip install eegdash`
# Habilita visualização gráfica inline no Jupyter
%matplotlib inline

# Compartilhar o pré-processamento espectral com uma árvore de características

Compare os cálculos de Welch separados e compartilhados nos mesmos ensaios gravados.

Essas gravações reais de SSVEP de Nakanishi2015 são distribuídas como a versão processada
[nm000118](https://nemar.org/dataset/nm000118)
([estudo](https://doi.org/10.1371/journal.pone.0140703)).
Filtragem, redução de taxa de amostragem (*downsampling*) e tratamento de latência já foram aplicados;
não desloque os inícios dos eventos novamente. CPU é suficiente. Conexão com a Internet é necessária
para o primeiro download; ``EEGDASH_CACHE_DIR`` mantém os downloads entre as execuções.

O subconjunto explícito usa 1 participante, cerca de 7.0 MB de arquivos de sinal.

## Antes de começar
Instale o EEGDash e suas dependências. O tutorial 40 apresenta a API do extrator espectral;
esta página reconstrói as janelas de um único sujeito e não precisa de uma tabela de características salva.
O resultado final é uma verificação de equivalência e a medição do tempo gasto para duas maneiras
de expressar as mesmas características de banda.


In [ ]:
# Importa utilitários de sistema e de caminhos no sistema de arquivos
import os
from pathlib import Path

# Importa bibliotecas para plotagem, manipulação de arrays e análise tabular
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
# Importa função para fatiar o sinal contínuo em janelas de eventos da Braindecode
from braindecode.preprocessing import create_windows_from_events

# Importa classes de dataset e extratores espectrais do EEGDash
from eegdash import EEGDashDataset
from functools import partial
from eegdash.features import (
    FeatureExtractor,
    extract_features,
    spectral_bands_power,
    spectral_preprocessor,
)
# Importa temporizador de alta resolução para medição de performance
from time import perf_counter

## 1. Carregar e inspecionar as gravações selecionadas
Use uma única gravação para que ambos os caminhos de extração vejam exatamente a mesma
ordem de canais, taxa de amostragem e rótulos de eventos. O tempo de download fica fora
da região cronometrada. A comparação diz respeito ao cálculo das características,
não ao desempenho de rede.



In [ ]:
# Define o diretório de cache baseado em variável de ambiente ou valor padrão
cache_dir = Path(os.environ.get("EEGDASH_CACHE_DIR", ".eegdash_cache"))
# Seleciona apenas o sujeito 1 para teste de tempo e equivalência
subjects = ["1"]
# Inicializa e carrega a gravação do sujeito 1 na sessão 0 e run 0 da tarefa de SSVEP
dataset = EEGDashDataset(
    cache_dir=cache_dir,
    dataset="nm000118",
    subject=subjects,
    session="0",
    run="0",
    task="ssvep",
    n_jobs=1,
)
# Confirma que exatamente uma gravação foi carregada
assert len(dataset.datasets) == len(subjects)
# Exibe resumo da descrição do sujeito
print(dataset.description[["subject", "session", "run"]])
# Acessa os dados brutos e extrai a taxa de amostragem e canais
raw = dataset.datasets[0].raw
sfreq = raw.info["sfreq"]
channel_names = raw.ch_names
# Ordena as anotações de frequência de estímulo numericamente
class_names = sorted(set(raw.annotations.description), key=float)
# Cria o mapeamento de classes para índices numéricos de 0 a 11
mapping = {name: index for index, name in enumerate(class_names)}
# Assegura a presença de 12 frequências no mapeamento
assert len(mapping) == 12
# Verifica conformidade de canais, frequência de amostragem e anotações
for recording in dataset.datasets:
    assert recording.raw.ch_names == channel_names
    assert recording.raw.info["sfreq"] == sfreq
    assert set(recording.raw.annotations.description) == set(mapping)
# Imprime informações dos canais e classes observadas
print(f"Channels: {channel_names}; sampling rate: {sfreq} Hz")
print("Observed stimulus frequencies (Hz):", class_names)

## 2. Janelar os ensaios observados
Mantenha o contrato de janela de quatro segundos do tutorial 40. Cada extrator lê os mesmos
arrays de ensaio por meio do Braindecode; alterar as janelas entre os caminhos misturaria
uma mudança nos dados com a comparação computacional.
O evento original dura 4.15 segundos; seus 0.15 segundos finais não são usados.



In [ ]:
# Define o tamanho da janela para 4 segundos em amostras (1024 amostras a 256 Hz)
window_size = int(4 * sfreq)
# Cria janelas a partir dos eventos descartando a fração excedente final
windows = create_windows_from_events(
    dataset,
    mapping=mapping,
    window_size_samples=window_size,
    window_stride_samples=window_size,
    on_last_window="drop",
    preload=True,
)
# Obtém os metadados das janelas com índice reiniciado
metadata = windows.get_metadata().reset_index(drop=True)
# Extrai o array numpy dos alvos (classes de 0 a 11)
y = metadata["target"].to_numpy(dtype=int)
# Assegura contagens e classes consistentes
assert len(windows) == len(metadata)
assert set(y) == set(mapping.values())
# Confirma unicidade das janelas criadas
assert not metadata.duplicated(["subject", "session", "run", "i_start_in_trial"]).any()
# Exibe tabela cruzada de distribuição de classes para o sujeito
print(pd.crosstab(metadata["subject"], y))

## 3. Construir características espectrais equivalentes: independentes vs. compartilhadas
Cada ``partial`` fixa os limites de uma banda. O dicionário simples (*flat*) envolve cada
banda em seu próprio pré-processador espectral, enquanto a árvore coloca o pré-processador
comum acima de todas as folhas de banda. Esse pai compartilhado é a razão pela qual o espectro
de Welch pode ser reutilizado; colocar nomes semelhantes em um dicionário sozinho não eliminaria
o pré-processamento repetido.

Ambas as definições usam os mesmos segmentos Welch de um segundo e faixa de 4–30 Hz.
Os nomes aninhados produzem prefixos de coluna diferentes, portanto a verificação de igualdade abaixo
compara cada par semântico banda/canal explicitamente. Formatos de tabela idênticos não provariam
valores de características idênticos.



In [ ]:
# Dicionário com as definições dos intervalos das bandas: teta, alfa e beta
bands = {"theta": (4, 8), "alpha": (8, 12), "beta": (12, 30)}
# Configura a função parcial do pré-processador PSD de Welch com janela de 1s de 4 a 30 Hz
psd = partial(spectral_preprocessor, fs=sfreq, nperseg=int(sfreq), f_min=4, f_max=30)
# Cria as funções de cálculo para cada banda individual como folhas da árvore
leaves = {
    name: partial(spectral_bands_power, bands={name: limits})
    for name, limits in bands.items()
}
# Estrutura 'flat': cada banda recalcula seu próprio espectro de Welch separadamente
flat = {
    name: FeatureExtractor({"power": leaf}, preprocessor=psd)
    for name, leaf in leaves.items()
}
# Estrutura 'tree': um único pré-processador calcula o espectro uma única vez e o compartilha entre as bandas
tree = FeatureExtractor(leaves, preprocessor=psd)
# Imprime a representação da árvore de características
print(tree)

## 4. Medir ambos os caminhos e verificar os valores das características, não uma aceleração prometida
O temporizador envolve a operação completa ``extract_features(...).to_dataframe()``,
incluindo processamento em lotes e montagem da tabela. Uma razão acima de um significa que
a árvore compartilhada foi mais rápida nesta execução; abaixo de um significa que foi mais lenta.
Gravações pequenas podem ser dominadas por sobrecargas (*overheads*) fixas do framework, e a segunda
medição pode se beneficiar de caches aquecidos.

A asserção ``assert_allclose`` é o resultado de correção. As duas barras medidas são a observação
de desempenho, sem impor um limite inferior à aceleração. A tolerância numérica considera
pequenas variações de ponto flutuante enquanto ainda verifica se cada banda e canal nomeados
possuem valores equivalentes.



In [ ]:
# Cronometra a extração no formato separado (flat)
start = perf_counter()
flat_table = extract_features(windows, flat, batch_size=64, n_jobs=1).to_dataframe()
flat_seconds = perf_counter() - start
# Cronometra a extração no formato de árvore compartilhada (tree)
start = perf_counter()
tree_table = extract_features(windows, tree, batch_size=64, n_jobs=1).to_dataframe()
tree_seconds = perf_counter() - start
# Valida numericamente que cada banda e canal apresentam valores equivalentes entre os dois métodos
for band in bands:
    for channel in channel_names:
        np.testing.assert_allclose(
            flat_table[f"{band}_power_{band}_{channel}"],
            tree_table[f"{band}_{band}_{channel}"],
            rtol=1e-6,
        )
# Garante que todos os valores obtidos na árvore são finitos
assert np.isfinite(tree_table.to_numpy()).all()
# Imprime os tempos de execução medidos e a razão entre eles
print(
    f"Flat: {flat_seconds:.3f}s; tree: {tree_seconds:.3f}s; ratio: {flat_seconds / tree_seconds:.2f}"
)
# Gera gráfico de barras comparando os tempos de processamento medidos
fig, ax = plt.subplots(figsize=(6, 3), layout="constrained")
ax.bar(["Separate spectra", "Shared spectrum"], [flat_seconds, tree_seconds])
ax.set(ylabel="Measured extraction time (s)", title="Same trials and band powers")
# Exibe o gráfico comparativo de tempo
plt.show()

## Estender a árvore e verificar a equivalência novamente
Adicione a mesma banda adicional a ``bands`` e execute novamente ambos os caminhos. A tabela
deve ganhar uma coluna por canal, e o loop de equivalência a cobrirá automaticamente.
Para um estudo de desempenho mais rigoroso, repita as medições em uma gravação representativa e
alterne a ordem de execução; uma única execução não deve ser relatada como uma referência geral de velocidade.

